# GeoSharp — LDSR-S2 Fine-Tuning on WorldStrat x4

**Confirmed WorldStrat x4 data**
- LR: `4 × 128 × 128` at 10 m
- HR: `4 × 512 × 512` at 2.5 m
- Bands: B02, B03, B04, B08

The 512×512 HR target is used directly. No 256→512 artificial upsampling is performed.


In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA available: False


c:\Users\Raaghav\miniconda3\envs\ncl_sae\lib\site-packages\torch\cuda\__init__.py:180: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 1: invalid argument (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [ ]:
# Configuration

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "datasets" / "datasets" / "worldstrat_x4"

INDEX_PATH = PROJECT_ROOT / "data" / "index.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "finetuned"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 1
NUM_WORKERS = 0

EPOCHS = 5

LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_ACCUM_STEPS = 2

MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None

SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DATA_DIR:", DATA_DIR)
print("INDEX_PATH:", INDEX_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("DEVICE:", DEVICE)

DATA_DIR: c:\Users\Raaghav\Desktop\coding\sih\prototype\data\datasets\datasets\worldstrat_x4
INDEX_PATH: c:\Users\Raaghav\Desktop\coding\sih\prototype\data\index.csv
OUTPUT_DIR: c:\Users\Raaghav\Desktop\coding\sih\prototype\outputs\finetuned
DEVICE: cpu


In [3]:
# Verify data paths

assert DATA_DIR.exists(), f"WorldStrat directory not found: {DATA_DIR}"
assert INDEX_PATH.exists(), f"index.csv not found: {INDEX_PATH}"

print("✓ WorldStrat data found")
print("✓ index.csv found")


✓ WorldStrat data found
✓ index.csv found


In [4]:
# Load WorldStrat index.csv

index_df = pd.read_csv(INDEX_PATH, sep="\t")

print("Rows:", len(index_df))
print("Columns:", list(index_df.columns))
print("\nSplit counts:")
print(index_df["split"].value_counts())


Rows: 3132
Columns: ['tile', 'Unnamed: 0', 'IPCC Class', 'SMOD', 'source', 'joint_class', 'split', 'hr_img', 'lr_img', 'correlation']

Split counts:
split
train    2503
val       318
test      311
Name: count, dtype: int64


In [5]:
# Dataset

def read_tiff(path):
    with rasterio.open(path) as src:
        return src.read()


def normalize_reflectance(arr):
    arr = np.nan_to_num(
        arr.astype(np.float32),
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    if np.nanmax(arr) > 2.0:
        arr = arr / 10000.0

    return np.clip(arr, 0.0, 1.0)


class WorldStratX4Dataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]

        lr_path = DATA_DIR / str(row["lr_img"]).strip()
        hr_path = DATA_DIR / str(row["hr_img"]).strip()

        if not lr_path.exists():
            raise FileNotFoundError(lr_path)

        if not hr_path.exists():
            raise FileNotFoundError(hr_path)

        lr = normalize_reflectance(read_tiff(lr_path))
        hr = normalize_reflectance(read_tiff(hr_path))

        if lr.shape != (4, 128, 128):
            raise ValueError(
                f"Expected LR (4,128,128), got {lr.shape} for {lr_path.name}"
            )

        if hr.shape != (4, 512, 512):
            raise ValueError(
                f"Expected HR (4,512,512), got {hr.shape} for {hr_path.name}"
            )

        return {
            "lr": torch.from_numpy(lr),
            "hr": torch.from_numpy(hr),
            "name": lr_path.name,
        }


train_frame = index_df[
    index_df["split"].astype(str).str.lower() == "train"
].copy()

val_frame = index_df[
    index_df["split"].astype(str).str.lower() == "val"
].copy()

test_frame = index_df[
    index_df["split"].astype(str).str.lower() == "test"
].copy()

if MAX_TRAIN_SAMPLES is not None:
    train_frame = train_frame.head(MAX_TRAIN_SAMPLES)

if MAX_VAL_SAMPLES is not None:
    val_frame = val_frame.head(MAX_VAL_SAMPLES)

train_ds = WorldStratX4Dataset(train_frame)
val_ds = WorldStratX4Dataset(val_frame)

print("Train samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Held-out test samples (NOT USED):", len(test_frame))

sample = train_ds[0]

print("LR:", tuple(sample["lr"].shape))
print("HR:", tuple(sample["hr"].shape))
print("Sample:", sample["name"])


Train samples: 500
Validation samples: 20
Held-out test samples (NOT USED): 311
LR: (4, 128, 128)
HR: (4, 512, 512)
Sample: Landcover-1174386_lr.tif


In [6]:
# DataLoaders

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


Train batches: 500
Validation batches: 20


In [7]:
# Reproducibility

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)


Seed: 42


In [8]:
# Load pretrained ESA OpenSR LDSR-S2
# This is the SAME config-loading mechanism used by the working GeoSharp inference.py.

import requests
from io import StringIO
from omegaconf import OmegaConf
import opensr_model

config_url = (
    "https://raw.githubusercontent.com/"
    "ESAOpenSR/opensr-model/"
    "refs/heads/main/"
    "opensr_model/configs/config_10m.yaml"
)

response = requests.get(config_url, timeout=30)
response.raise_for_status()

config = OmegaConf.load(StringIO(response.text))

model = opensr_model.SRLatentDiffusion(
    config,
    device=DEVICE
)

# The official loader downloads the checkpoint automatically if needed.
model.load_pretrained(config.ckpt_version)
model = model.to(DEVICE)

print("✓ LDSR-S2 model loaded successfully.")


W0906 15:58:33.637000 5056 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


LatentDiffusion: Running in eps-prediction mode
DiffusionWrapper has 113.63 M params.
Keeping EMAs of 308.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 128, 128) = 65536 dimensions.
making attention of type 'vanilla' with 512 in_channels
Normalization disabled.
Loaded pretrained weights from:  opensr-ldsrs2_v1_0_0.ckpt
✓ LDSR-S2 model loaded successfully.


In [9]:
# Freeze autoencoder / conditioning and fine-tune diffusion U-Net

for p in model.model.first_stage_model.parameters():
    p.requires_grad = False

if hasattr(model.model, "cond_stage_model") and model.model.cond_stage_model is not None:
    for p in model.model.cond_stage_model.parameters():
        p.requires_grad = False

trainable_params = []

for p in model.model.model.diffusion_model.parameters():
    p.requires_grad = True
    trainable_params.append(p)

print("Trainable parameters:", f"{sum(p.numel() for p in trainable_params):,}")
print("Total parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


Trainable parameters: 113,626,884
Total parameters: 168,955,116


In [10]:
# Sanity check

batch = next(iter(train_loader))

print("LR batch:", tuple(batch["lr"].shape))
print("HR batch:", tuple(batch["hr"].shape))

assert tuple(batch["lr"].shape[1:]) == (4, 128, 128)
assert tuple(batch["hr"].shape[1:]) == (4, 512, 512)

print("✓ 10m LR: 128×128")
print("✓ 2.5m HR: 512×512")
print("✓ Ready for direct 10m → 2.5m training")


LR batch: (1, 4, 128, 128)
HR batch: (1, 4, 512, 512)
✓ 10m LR: 128×128
✓ 2.5m HR: 512×512
✓ Ready for direct 10m → 2.5m training


In [11]:
# Encoding + diffusion training loss

# IMPORTANT:
# HR is already 512×512 @ 2.5m.
# Do NOT resize HR.

def encode_hr(model, hr):
    hr_norm = model.linear_transform(hr, stage="norm")
    posterior = model.model.first_stage_model.encode(hr_norm)
    return model.model.get_first_stage_encoding(posterior)


def encode_condition(model, lr):
    # LR is placed on the 512×512 model grid as conditioning.
    if model.encode_conditioning:
        lr_up = F.interpolate(
            lr,
            size=(512, 512),
            mode="bilinear",
            align_corners=False
        )

        lr_norm = model.linear_transform(lr_up, stage="norm")

        return model.model.first_stage_model.encode(
            lr_norm
        ).sample()

    return model.linear_transform(lr, stage="norm")


def diffusion_training_loss(model, lr, hr):
    diffusion = model.model

    # Direct 2.5m HR target.
    z_hr = encode_hr(model, hr)

    # 10m LR conditioning.
    condition = encode_condition(model, lr)

    batch_size = z_hr.shape[0]

    timestep = torch.randint(
        0,
        diffusion.num_timesteps,
        (batch_size,),
        device=DEVICE
    ).long()

    noise = torch.randn_like(z_hr)

    noisy_latent = diffusion.q_sample(
        z_hr,
        timestep,
        noise=noise
    )

    predicted_noise = diffusion.apply_model(
        noisy_latent,
        timestep,
        condition
    )

    return F.mse_loss(predicted_noise, noise)


In [12]:
# Optimizer

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print("Optimizer ready.")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)


Optimizer ready.
Learning rate: 1e-05
Weight decay: 0.0001
Gradient accumulation: 2


In [13]:
# Training + validation

best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):

    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_sum = 0.0

    for step, batch in enumerate(train_loader, start=1):

        lr = batch["lr"].to(DEVICE, non_blocking=True)
        hr = batch["hr"].to(DEVICE, non_blocking=True)

        loss = diffusion_training_loss(model, lr, hr)

        (loss / GRAD_ACCUM_STEPS).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        train_sum += loss.item()

        if step % 100 == 0 or step == len(train_loader):
            print(
                f"Epoch {epoch}/{EPOCHS} | "
                f"Step {step}/{len(train_loader)} | "
                f"Loss {loss.item():.6f}"
            )

    train_loss = train_sum / max(1, len(train_loader))

    # Validation
    model.eval()
    val_sum = 0.0

    with torch.no_grad():
        for batch in val_loader:
            lr = batch["lr"].to(DEVICE, non_blocking=True)
            hr = batch["hr"].to(DEVICE, non_blocking=True)

            val_loss = diffusion_training_loss(model, lr, hr)
            val_sum += val_loss.item()

    val_loss = val_sum / max(1, len(val_loader))

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
    })

    print(
        f"\nEpoch {epoch}/{EPOCHS} complete | "
        f"Train: {train_loss:.6f} | Val: {val_loss:.6f}"
    )

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_path = OUTPUT_DIR / "ldsrs2_worldstrat_x4_best.pt"

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss,
                "train_loss": train_loss,
                "dataset": "WorldStrat x4",
                "input_shape": (4, 128, 128),
                "target_shape": (4, 512, 512),
                "input_resolution_m": 10.0,
                "target_resolution_m": 2.5,
            },
            best_path,
        )

        print("Saved BEST:", best_path)

print("\nTraining finished.")


Epoch 1/1 | Step 100/500 | Loss 0.015262
Epoch 1/1 | Step 200/500 | Loss 0.369855
Epoch 1/1 | Step 300/500 | Loss 0.175710
Epoch 1/1 | Step 400/500 | Loss 0.370299
Epoch 1/1 | Step 500/500 | Loss 0.169244

Epoch 1/1 complete | Train: 0.343669 | Val: 0.302710
Saved BEST: c:\Users\Raaghav\Desktop\coding\sih\prototype\outputs\finetuned\ldsrs2_worldstrat_x4_best.pt

Training finished.


In [14]:
from pathlib import Path

for p in [
    Path("outputs/finetuned/ldsrs2_worldstrat_x4_best.pt"),
    Path("outputs/finetuned/ldsrs2_worldstrat_x4_finetuned.pt"),
]:
    print(p, "→", p.exists(), p.stat().st_size / 1024**2 if p.exists() else "-")

outputs\finetuned\ldsrs2_worldstrat_x4_best.pt → True 1945.4950170516968
outputs\finetuned\ldsrs2_worldstrat_x4_finetuned.pt → False -


In [15]:
# Save final checkpoint

final_path = OUTPUT_DIR / "ldsrs2_worldstrat_x4_finetuned.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": EPOCHS,
        "best_val_loss": best_val_loss,
        "history": history,
        "dataset": "WorldStrat x4",
        "input_shape": (4, 128, 128),
        "target_shape": (4, 512, 512),
        "input_resolution_m": 10.0,
        "target_resolution_m": 2.5,
    },
    final_path,
)

print("Final checkpoint saved:", final_path)
print("Best validation loss:", best_val_loss)
print(pd.DataFrame(history))


Final checkpoint saved: c:\Users\Raaghav\Desktop\coding\sih\prototype\outputs\finetuned\ldsrs2_worldstrat_x4_finetuned.pt
Best validation loss: 0.30271036168560383
   epoch  train_loss  val_loss
0      1    0.343669   0.30271


In [16]:
# Final checkpoint verification

best_path = OUTPUT_DIR / "ldsrs2_worldstrat_x4_best.pt"
final_path = OUTPUT_DIR / "ldsrs2_worldstrat_x4_finetuned.pt"

assert best_path.exists()
assert final_path.exists()

print("✓ BEST checkpoint:", best_path)
print("✓ FINAL checkpoint:", final_path)
print("✓ Fine-tuning completed successfully.")


✓ BEST checkpoint: c:\Users\Raaghav\Desktop\coding\sih\prototype\outputs\finetuned\ldsrs2_worldstrat_x4_best.pt
✓ FINAL checkpoint: c:\Users\Raaghav\Desktop\coding\sih\prototype\outputs\finetuned\ldsrs2_worldstrat_x4_finetuned.pt
✓ Fine-tuning completed successfully.


In [2]:
# ============================================================
# STANDALONE TEST: BEST CHECKPOINT + PSNR/SSIM
# ============================================================

import numpy as np
import pandas as pd
import rasterio
import torch
import requests

from pathlib import Path
from io import StringIO
from omegaconf import OmegaConf
from torch.utils.data import Dataset
from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

import opensr_model


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "datasets" / "datasets" / "worldstrat_x4"
INDEX_PATH = PROJECT_ROOT / "data" / "index.csv"
BEST_PATH = PROJECT_ROOT / "outputs" / "finetuned" / "ldsrs2_worldstrat_x4_best.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Best checkpoint:", BEST_PATH)
print("Checkpoint exists:", BEST_PATH.exists())


# ------------------------------------------------------------
# 2. LOAD TEST INDEX
# ------------------------------------------------------------

index_df = pd.read_csv(INDEX_PATH, sep="\t")

test_frame = index_df[
    index_df["split"].astype(str).str.lower() == "test"
].copy()

# FIRST TEST RUN: only 5 images
test_frame = test_frame.head(5).reset_index(drop=True)

print("Test samples:", len(test_frame))


# ------------------------------------------------------------
# 3. TIFF + NORMALIZATION
# ------------------------------------------------------------

def read_tiff(path):
    with rasterio.open(path) as src:
        return src.read()


def normalize_reflectance(arr):
    arr = arr.astype(np.float32)

    if np.nanmax(arr) > 2.0:
        arr = arr / 10000.0

    return np.clip(arr, 0.0, 1.0)


# ------------------------------------------------------------
# 4. DATASET
# ------------------------------------------------------------

class WorldStratTestDataset(Dataset):

    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):

        row = self.frame.iloc[idx]

        lr_path = DATA_DIR / str(row["lr_img"]).strip()
        hr_path = DATA_DIR / str(row["hr_img"]).strip()

        lr = normalize_reflectance(read_tiff(lr_path))
        hr = normalize_reflectance(read_tiff(hr_path))

        return {
            "lr": torch.from_numpy(lr).float(),
            "hr": torch.from_numpy(hr).float(),
            "name": str(row["tile"])
        }


test_ds = WorldStratTestDataset(test_frame)

sample = test_ds[0]

print("LR shape:", tuple(sample["lr"].shape))
print("HR shape:", tuple(sample["hr"].shape))
print("Sample:", sample["name"])


# ------------------------------------------------------------
# 5. LOAD ORIGINAL ESA MODEL
# ------------------------------------------------------------

config_url = (
    "https://raw.githubusercontent.com/"
    "ESAOpenSR/opensr-model/"
    "refs/heads/main/"
    "opensr_model/configs/config_10m.yaml"
)

response = requests.get(config_url, timeout=30)
response.raise_for_status()

config = OmegaConf.load(StringIO(response.text))

model = opensr_model.SRLatentDiffusion(
    config,
    device=DEVICE
)

model.load_pretrained(config.ckpt_version)
model = model.to(DEVICE)


# ------------------------------------------------------------
# 6. LOAD OUR FINE-TUNED BEST CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    BEST_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("✓ Fine-tuned BEST checkpoint loaded")
print("Validation loss:", checkpoint.get("val_loss", "N/A"))


# ------------------------------------------------------------
# 7. SR FUNCTION
# ------------------------------------------------------------

def generate_sr(lr):

    lr = lr.unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        sr = model.forward(
            lr,
            sampling_steps=30
        )

    sr = sr.squeeze(0).cpu().numpy()

    # CHW -> HWC
    sr = np.transpose(sr, (1, 2, 0))

    return np.clip(sr, 0.0, 1.0)


# ------------------------------------------------------------
# 8. BICUBIC BASELINE
# ------------------------------------------------------------

def bicubic_sr(lr):

    lr = np.transpose(lr, (1, 2, 0))

    baseline = resize(
        lr,
        (512, 512, 4),
        order=3,
        mode="reflect",
        anti_aliasing=False,
        preserve_range=True
    )

    return np.clip(baseline, 0.0, 1.0)


# ------------------------------------------------------------
# 9. EVALUATION
# ------------------------------------------------------------

geo_psnr = []
geo_ssim = []

bic_psnr = []
bic_ssim = []


for i in range(len(test_ds)):

    sample = test_ds[i]

    lr = sample["lr"]
    hr = sample["hr"].numpy()

    # HR CHW -> HWC
    hr = np.transpose(hr, (1, 2, 0))
    hr = np.clip(hr, 0.0, 1.0)

    # GeoSharp
    sr = generate_sr(lr)

    # Bicubic
    bic = bicubic_sr(lr.numpy())

    # PSNR
    p_geo = peak_signal_noise_ratio(
        hr,
        sr,
        data_range=1.0
    )

    p_bic = peak_signal_noise_ratio(
        hr,
        bic,
        data_range=1.0
    )

    # SSIM
    s_geo = structural_similarity(
        hr,
        sr,
        channel_axis=-1,
        data_range=1.0
    )

    s_bic = structural_similarity(
        hr,
        bic,
        channel_axis=-1,
        data_range=1.0
    )

    geo_psnr.append(p_geo)
    geo_ssim.append(s_geo)

    bic_psnr.append(p_bic)
    bic_ssim.append(s_bic)

    print(
        f"{i+1}/{len(test_ds)} | "
        f"GeoSharp PSNR: {p_geo:.3f} dB | "
        f"SSIM: {s_geo:.4f} | "
        f"Bicubic PSNR: {p_bic:.3f} dB | "
        f"SSIM: {s_bic:.4f}"
    )


# ------------------------------------------------------------
# 10. FINAL RESULTS
# ------------------------------------------------------------

geo_psnr_mean = np.mean(geo_psnr)
geo_ssim_mean = np.mean(geo_ssim)

bic_psnr_mean = np.mean(bic_psnr)
bic_ssim_mean = np.mean(bic_ssim)

print("\n" + "=" * 60)
print("FINAL RESULTS — 5 TEST IMAGES")
print("=" * 60)

print(f"GeoSharp PSNR : {geo_psnr_mean:.3f} dB")
print(f"GeoSharp SSIM : {geo_ssim_mean:.4f}")

print()

print(f"Bicubic PSNR  : {bic_psnr_mean:.3f} dB")
print(f"Bicubic SSIM  : {bic_ssim_mean:.4f}")

print()

print(f"PSNR Gain     : {geo_psnr_mean - bic_psnr_mean:+.3f} dB")
print(f"SSIM Gain     : {geo_ssim_mean - bic_ssim_mean:+.4f}")

W0906 19:19:41.047000 18056 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Device: cuda
Best checkpoint: c:\Users\Raaghav\Desktop\coding\sih\prototype\outputs\finetuned\ldsrs2_worldstrat_x4_best.pt
Checkpoint exists: True
Test samples: 5
LR shape: (4, 128, 128)
HR shape: (4, 512, 512)
Sample: UNHCR-DJIs000145
LatentDiffusion: Running in eps-prediction mode
DiffusionWrapper has 113.63 M params.
Keeping EMAs of 308.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 128, 128) = 65536 dimensions.
making attention of type 'vanilla' with 512 in_channels
Normalization disabled.
Loaded pretrained weights from:  opensr-ldsrs2_v1_0_0.ckpt
✓ Fine-tuned BEST checkpoint loaded
Validation loss: 0.30271036168560383
1/5 | GeoSharp PSNR: 28.848 dB | SSIM: 0.6801 | Bicubic PSNR: 30.432 dB | SSIM: 0.8218
2/5 | GeoSharp PSNR: 11.213 dB | SSIM: 0.1783 | Bicubic PSNR: 11.178 dB | SSIM: 0.3658
3/5 | GeoSharp PSNR: 25.402 dB | SSIM: 0.7703 | Bicubic PSNR: 25.472 dB | SSIM: 0.8364
4/5 | GeoSharp PSNR: 29.921 dB | SSIM: 0.8665 | Bicubic PSNR: 30.22